# 02 — Preprocess LEMON fMRI & Extract Features

**Input dataset:** `lemon-meta` (attach it at `/kaggle/input/lemon-meta/`)

For each subject:
1. Download raw 4D BOLD from OpenNeuro S3 (~300 MB)
2. Compute **mean BOLD** volume (average over time)
3. Compute **fALFF** map (0.01–0.08 Hz power fraction)
4. Resample both to 48×48×32 MNI space
5. Save `(2, 48, 48, 32)` array → delete raw file

**After running:** Save output as Kaggle Dataset `lemon-features`.

> Kaggle sessions last 12 hours. With ~200 subjects at ~90 sec each,
> total runtime is ~5 hours — fits comfortably in one session.
> The loop skips already-processed subjects so you can re-run safely.

In [ ]:
!pip install -q nibabel nilearn scipy tqdm awscli

In [ ]:
import os, json, subprocess
import numpy as np

# Input: lemon-meta dataset attached by user
META_DIR = '/kaggle/input/lemon-meta'
FEAT_DIR = '/kaggle/working/features'
RAW_DIR  = '/kaggle/working/raw'
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(RAW_DIR,  exist_ok=True)

with open(f'{META_DIR}/subject_list.json') as f:
    subjects = json.load(f)

with open(f'{META_DIR}/path_config.json') as f:
    path_cfg = json.load(f)

BOLD_SUFFIX = path_cfg['bold_suffix']    # e.g. '_task-rest_bold.nii.gz'
FUNC_SUBDIR = path_cfg['func_subdir']    # e.g. 'func/' or ''
BUCKET      = 's3://openneuro.org/ds000221'

print(f'Subjects to process: {len(subjects)}')
print(f'BOLD suffix:  {BOLD_SUFFIX}')
print(f'Func subdir:  {FUNC_SUBDIR}')

In [ ]:
from scipy.signal import detrend

def compute_falff(bold_data, tr, low=0.01, high=0.08):
    """
    bold_data : (x, y, z, t)
    Returns   : (x, y, z) fALFF map
    """
    freqs    = np.fft.rfftfreq(bold_data.shape[3], d=tr)
    fft_data = np.abs(np.fft.rfft(bold_data, axis=3))
    low_mask = (freqs >= low) & (freqs <= high)
    total    = fft_data.sum(axis=3) + 1e-8
    low_pow  = fft_data[..., low_mask].sum(axis=3)
    return (low_pow / total).astype(np.float32)

print('fALFF function ready ✓')

In [ ]:
import nibabel as nib
from nilearn import image as nli

TARGET_SHAPE  = (48, 48, 32)
TARGET_AFFINE = np.diag([4.0, 4.0, 4.5, 1.0])

def preprocess_subject(bold_path):
    img = nib.load(bold_path)
    tr  = float(img.header.get_zooms()[3]) or 1.4
    data = img.get_fdata(dtype=np.float32)        # (x, y, z, t)

    # Mean BOLD
    mean_img = nib.Nifti1Image(data.mean(axis=3), img.affine)

    # fALFF on detrended signal
    falff     = compute_falff(detrend(data, axis=3, type='linear'), tr)
    falff_img = nib.Nifti1Image(falff, img.affine)

    # Resample to MNI target
    mean_r  = nli.resample_img(mean_img,  target_affine=TARGET_AFFINE, target_shape=TARGET_SHAPE)
    falff_r = nli.resample_img(falff_img, target_affine=TARGET_AFFINE, target_shape=TARGET_SHAPE)

    def norm(arr):
        m, s = arr.mean(), arr.std()
        return ((arr - m) / (s + 1e-8)).astype(np.float32)

    ch0 = norm(mean_r.get_fdata(dtype=np.float32))
    ch1 = norm(falff_r.get_fdata(dtype=np.float32))
    return np.stack([ch0, ch1], axis=0)   # (2, 48, 48, 32)

print('Preprocessing function ready ✓')

In [ ]:
from tqdm import tqdm
import time

failed  = []
skipped = 0
start   = time.time()

for sub in tqdm(subjects, desc='Preprocessing'):
    out_path = f'{FEAT_DIR}/{sub}.npy'
    if os.path.exists(out_path):   # resume-safe
        skipped += 1
        continue

    raw_path = f'{RAW_DIR}/{sub}_bold.nii.gz'
    bold_key = f'{BUCKET}/{sub}/{FUNC_SUBDIR}{sub}{BOLD_SUFFIX}'
    try:
        # Download using the path discovered by notebook 01
        r = subprocess.run(
            ['aws', 's3', 'cp', bold_key, raw_path, '--no-sign-request'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            raise RuntimeError(r.stderr)

        # Preprocess & save
        np.save(out_path, preprocess_subject(raw_path))

    except Exception as e:
        print(f'\nFAILED {sub}: {e}')
        failed.append(sub)
    finally:
        if os.path.exists(raw_path):
            os.remove(raw_path)

elapsed = (time.time() - start) / 60
done    = len(subjects) - len(failed) - skipped
print(f'\nDone in {elapsed:.0f} min — processed: {done}  skipped: {skipped}  failed: {len(failed)}')
if failed:
    with open('/kaggle/working/failed.json', 'w') as f:
        json.dump(failed, f)

In [ ]:
import matplotlib.pyplot as plt

feat_files = [f for f in os.listdir(FEAT_DIR) if f.endswith('.npy')]
print(f'Feature files: {len(feat_files)}')

sample = np.load(f'{FEAT_DIR}/{feat_files[0]}')
print(f'Shape: {sample.shape}  — expected (2, 48, 48, 32)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample[0, :, :, 16].T, cmap='gray', origin='lower')
axes[0].set_title('Mean BOLD — z=16')
axes[1].imshow(sample[1, :, :, 16].T, cmap='hot', origin='lower')
axes[1].set_title('fALFF — z=16')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_features.png', dpi=100)
plt.show()

print()
print('NEXT STEPS:')
print('  1. Save Version → tick "Save as Dataset" → name it "lemon-features"')
print('  2. Open notebook 03 and attach both lemon-meta and lemon-features')